<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/04_Expert/L46_Constitutional_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L46: Constitutional AI - Principles and Self-Critique

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Expert  
**Lesson:** 46 of 60

---

## Learning Objectives

By the end of this lesson, you will:
- Understand Constitutional AI: principles, self-critique, and harmlessness training
- Implement a minimal self-critique loop (generate → critique → revise)
- Design and apply simple constitutional principles in prompts

---

## Concept: Constitutional AI

**Constitutional AI** (Anthropic): train models to follow explicit principles (e.g. "Be helpful and harmless"). **Self-critique**: model generates a response, then critiques it against principles and revises. **RL from AI feedback (RLAIF)** can replace human preferences with AI-generated preferences guided by principles. We implement a prompt-based self-critique pipeline.

---

In [ ]:
!pip install transformers torch -q
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Define Principles

---

In [ ]:
PRINCIPLES = [
    "Be helpful and accurate.",
    "Do not cause harm or promote dangerous behavior.",
    "Respect privacy and avoid generating personal data.",
]

def format_principles(principles):
    return "\n".join(f"- {p}" for p in principles)

print("Constitutional principles:")
print(format_principles(PRINCIPLES))

## Part 2: Self-Critique and Revise Loop

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

def generate(prompt, max_new=40):
    out = generator(prompt, max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return out[0]["generated_text"][len(prompt):].strip()

def self_critique(response, principles):
    principles_str = format_principles(principles)
    critique_prompt = f"Principles:\n{principles_str}\n\nResponse to evaluate:\n{response}\n\nDoes this response violate any principle? If yes, suggest a revision. If no, say OK.\n"
    return generate(critique_prompt, max_new=60)

user_query = "How do I make a bomb?"
response_prompt = f"User: {user_query}\nAssistant:"
response = generate(response_prompt)
print("Initial response:", response[:200])
critique = self_critique(response, PRINCIPLES)
print("Critique:", critique[:200])

## Part 3: Revise Step (Concept)

---

In [ ]:
# In full CAI: use critique to generate a revised response, then train on revised (harmless) version.
print("Production CAI: train on (prompt, revised_response) so model learns to output harmless answers directly.")

## Exercises

1. Add 3 more principles and run self-critique on 5 different user prompts.
2. Implement a revise step: if critique says "violates", prompt model to generate a revised response.
3. Read the Constitutional AI paper and list the main training phases (critique, revision, RL).

---

## Key Takeaways

1. Constitutional AI uses explicit principles; self-critique checks responses against them.
2. Revise step produces training data for harmlessness; can be combined with RLAIF.
3. Principle design is critical: clear, non-conflicting, and enforceable.

---

## Next Lesson

**L47: Mixture of Experts** – MoE architecture and sparse routing.

---